In [ ]:
if BASE not in ("EXIOBASE", "FIGARO", "OECD_ICIO", "GTAP"):
    raise ValueError(f"BASE={BASE!r} : le choc utilise les flux Z d'une base MRIO (EXIOBASE, FIGARO, OECD_ICIO ou GTAP)")


Z_REF_YEAR = 2010   # year whose z_CI is used to rebuild the volumes of the years without z_CI (fixed technical coefficients)


def mpath(name, year, suffix="_agg"):
    return Path(SRC_DIR) / BASE / f"output_loop_{BASE}_{year}" / f"{name}_{BASE}_{year}{suffix}.csv"


prod = pd.read_csv(Path(SRC_DIR) / BASE / f"output_loop_{BASE}" / f"production_{BASE}_agg.csv", sep=";").set_index(["Region", "Sector"])
_cache = {}


def load_z(c, year):
    if mpath("z_CI", year).exists():
        return flat(read_matrix(mpath("z_CI", year))).reindex(index=c.index, columns=c.columns).fillna(0.0), "z_CI exact"
    if "A" not in _cache:
        for d in (SRC_DIR, PATH_WRK + "SRDV (c,s)/", PATH_WRK + "SRDQ (c,s)/"):   # z_CI of the reference year, in any run folder
            p = Path(d) / BASE / f"output_loop_{BASE}_{Z_REF_YEAR}" / f"z_CI_{BASE}_{Z_REF_YEAR}_agg.csv"
            if p.exists():
                z_ref = flat(read_matrix(p)).reindex(index=c.index, columns=c.columns).fillna(0.0)
                x_ref = prod[str(Z_REF_YEAR)].reindex(c.index).fillna(0.0).to_numpy()
                _cache["A"] = z_ref.to_numpy() / np.where(x_ref > 0, x_ref, 1.0)[None, :]
                break
        else:
            raise FileNotFoundError(f"z_CI_{BASE}_{Z_REF_YEAR}_agg.csv introuvable : lancer know_calc_loop4.0 pour l'année {Z_REF_YEAR}")
    x = prod[str(year)].reindex(c.index).fillna(0.0).to_numpy()
    z_prox = _cache["A"] * x[None, :]                                    # volumes with the technical coefficients of Z_REF_YEAR
    zs, cs = z_prox.sum(axis=1), c.to_numpy().sum(axis=1)
    rho = np.divide(cs, zs, out=np.zeros(len(zs)), where=zs > 0)         # R&D of the supplier per euro sold this year
    z = np.where(rho[:, None] > 0, np.divide(c.to_numpy(), rho[:, None], out=np.zeros(c.shape), where=rho[:, None] > 0), z_prox)   # z = c / rho
    return pd.DataFrame(z, index=c.index, columns=c.columns), f"z reconstruit (c/rho, A {Z_REF_YEAR})"


def find(name):
    for d in SRC_DIRS:
        p = Path(d) / BASE / f"output_loop_{BASE}_{YEAR}" / f"{name}_{BASE}_{YEAR}_agg.csv"
        if p.exists():
            return p
    raise FileNotFoundError(f"{name}_{BASE}_{YEAR}_agg.csv introuvable dans {SRC_DIRS} : lancer know_calc_loop puis aggregate_to_common_mapping")


inputs = Path(PATH_WRK) / "choc_tfp" / "inputs"
omega_all = pd.read_csv(inputs / OMEGA_FILE)                                      # ω of all the years (year column optional)

prefix = {"dir": "rd_dir_CI", "tot": "rd_tot_CI"}[CONTENT]
c0 = flat(read_matrix(find(prefix)))
if CHOC_SCENARIO == "B":                      # only scenario B needs the volumes z (A does not)
    z0, z_src = load_z(c0, YEAR)
else:
    z0, z_src = None, "scénario A : pas de volumes z"
print(f"volumes {YEAR} : {z_src}")

cd, cf = block_matrices(c0)                       # domestic / foreign content, aggregated by (block, sector) BEFORE the shock
zd, zf = (None, None) if z0 is None else [m.reindex(index=cd.index, columns=cd.columns).fillna(0.0) for m in block_matrices(z0)]
print(f"{cd.shape[0]} (bloc, secteur) fournisseurs x {cd.shape[1]} (bloc, secteur) acheteurs")

out = Path(RESULTS_DIR) / "choc_tfp"
out.mkdir(parents=True, exist_ok=True)


In [ ]:
# Table of the displayed years: one row per (scenario, sector, bloc), one column per year. Change in unit cost relative to the baseline
# (the base year does not move: 0)
tables = {}
for model in MODEL_BLOCKS:
    wide = results[model].pivot_table(index=["scenario", "sector", "bloc"], columns="year", values="dlnCost").reindex(columns=DISPLAY_YEARS)
    wide.insert(0, YEAR, 0.0)
    wide = (np.exp(wide) - 1) * 100
    wide.columns.name = None
    tables[model] = wide.round(4).reset_index()
    path = out / f"variation_cout_{BASE}_{CONTENT}_modele{model}_base{YEAR}.csv"
    tables[model].to_csv(path, index=False)
    print(f"modèle ({model}) : tableau de {len(tables[model])} lignes écrit dans {path}")

tables[2]